# <span style="color:darkblue"> Lecture 11d: (Optional) Panel Data Linear Models </span>

<font size = "5">

This lecture is targeted at advanced regression users familiar with panel models.

This lecture introduces `linearmodels`, a Python package for several econometric models that are not as convenient to fit with `statsmodels`—especially panel-data. It complements `statsmodels`.

Our goal will be to 
- Recognize panel data and organize it with a two-level index.
- Fit and interpret pooled, fixed-effects, and other panel estimators.


# <span style="color:darkblue"> I. Install and Import Libraries </span>

<font size = "5">

Install `linearmodels` from a terminal: <br>

- ``` conda install -c conda-forge linearmodels  ``` <br>
(see Lecture 0 for how to install new packages)


<font size = "5">

Import relevant packages

In [1]:
import pandas as pd
import statsmodels.formula.api as smf

from linearmodels.datasets import mroz, wage_panel
from linearmodels.panel import (
    BetweenOLS,
    FirstDifferenceOLS,
    PanelOLS,
    PooledOLS,
    RandomEffects,
)
from linearmodels.iv import IV2SLS

# <span style="color:darkblue"> II. What is Panel Data? </span>

<font size = "5">

Panel (or longitudinal) data follow the same units—people, firms, schools, or countries—over multiple periods. 
- A row is one unit-period observation.
- This structure lets us distinguish variation **between** units from changes **within** the same unit over time.



<font size = 5>

We will use the package's built-in `wage_panel` example data: 

<font size = 3>

- 545 men followed annually from 1980 to 1987.
- The outcome is log hourly wage (`lwage`); variables include experience (`exper`), experience squared (`expersq`), marital status (`married`), and union membership (`union`).




In [2]:
# Load the example data shipped with linearmodels.
panel = wage_panel.load()
print(wage_panel.DESCR)


F. Vella and M. Verbeek (1998), "Whose Wages Do Unions Raise? A Dynamic Model
of Unionism and Wage Rate Determination for Young Men," Journal of Applied
Econometrics 13, 163-183.

nr                       person identifier
year                     1980 to 1987
black                    =1 if black
exper                    labor market experience
hisp                     =1 if Hispanic
hours                    annual hours worked
married                  =1 if married
educ                     years of schooling
union                    =1 if in union
lwage                    log(wage)
expersq                  exper^2
occupation               Occupation code



<font size = 5>

`linearmodels` panel estimators accept a DataFrame indexed by entity and time. 

In [3]:
# Use the person identifier and year as the panel's two-level index.
panel["year_cat"] = panel["year"].astype("category")
panel = panel.set_index(["nr", "year"]).sort_index()



## The common workflow

<font size = "5">

Most models follow the same pattern: choose an estimator, describe the outcome and regressors with a formula, then call `.fit()`. The fitted result object contains estimates, uncertainty measures, tests, and a printable summary. Here, `PooledOLS` treats all rows as one pooled sample and does not control for persistent entity-specific effects.

$ Y_{it} = X_{it}'\beta + \text{Error}_{it} $,

where $i$ denotes an entity and $t$ a time period.

In [4]:
pooled_model = PooledOLS.from_formula(
    "lwage ~ married + union",
    data=panel
)
pooled_results = pooled_model.fit(
    cov_type="clustered", cluster_entity=True
)
print(pooled_results.summary)

                          PooledOLS Estimation Summary                          
Dep. Variable:                  lwage   R-squared:                        0.5418
Estimator:                  PooledOLS   R-squared (Between):              0.6490
No. Observations:                4360   R-squared (Within):              -1.8062
Date:                Mon, Oct 05 2026   R-squared (Overall):              0.5418
Time:                        22:22:39   Log-likelihood                   -6882.8
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      2576.1
Entities:                         545   P-value                           0.0000
Avg Obs:                       8.0000   Distribution:                  F(2,4358)
Min Obs:                       8.0000                                           
Max Obs:                       8.0000   F-statistic (robust):             2058.3
                            

# <span style="color:darkblue"> III. Fixed Effects with `PanelOLS` </span>

<font size = "5">

Entity fixed effects control for all **time-invariant** differences between workers, whether observed or not. The coefficient on `married`, for example, is identified from changes within the same worker over time—not simply from comparing workers with different marital status.

$ Y_{it} = X_{it}'\beta + \alpha_i + \gamma_t + \text{Error}_{it} $,

where $i$ denotes an entity, $t$ a time period, $\alpha_i$ denotes entity fixed effects, and $\gamma_t$ denote time fixed effects.

<font size = "3">

- `EntityEffects` and `TimeEffects` request worker and year fixed effects.
- Time-invariant regressors such as education and race are absorbed by worker effects and cannot be separately estimated in this specification, so the formula omits a separate intercept.
- Standard errors clustered by entity allow errors to be correlated within an entity over time. Clustering changes uncertainty estimates, not the coefficient estimates.

In [5]:
fixed_effects_model = PanelOLS.from_formula(
    "lwage ~ married + union + EntityEffects + TimeEffects",
    data=panel,
)
fixed_effects_results = fixed_effects_model.fit(
    cov_type="clustered", cluster_entity=True
)
print(fixed_effects_results.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                  lwage   R-squared:                        0.0076
Estimator:                   PanelOLS   R-squared (Between):              0.0543
No. Observations:                4360   R-squared (Within):               0.0230
Date:                Mon, Oct 05 2026   R-squared (Overall):              0.0530
Time:                        22:22:39   Log-likelihood                   -1355.7
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      14.640
Entities:                         545   P-value                           0.0000
Avg Obs:                       8.0000   Distribution:                  F(2,3806)
Min Obs:                       8.0000                                           
Max Obs:                       8.0000   F-statistic (robust):             8.3958
                            

# <span style="color:darkblue"> IV. Other Panel Estimators </span>

<font size = "5">

`linearmodels.panel` also offers estimators that target different sources of variation or make different assumptions:

- **BetweenOLS** relates entity-level averages of the outcome and regressors.
- **RandomEffects** models entity-specific effects as random and requires them to be uncorrelated with the regressors.
- **FirstDifferenceOLS** relates changes in the outcome to changes in regressors,  <br>  by obtaining the difference
$Y_{it} - Y_{i,t-1}$ within the same entity.

In [9]:
between_results = BetweenOLS.from_formula(
    "lwage ~ exper + expersq + married + union", data=panel
).fit()

random_effects_results = RandomEffects.from_formula(
    "lwage ~ exper + expersq + married + union", data=panel
).fit()

first_difference_results = FirstDifferenceOLS.from_formula(
    "lwage ~  exper + expersq + married + union", data=panel
).fit()

print("Between OLS coefficients:")
print(between_results.params)
print("\nRandom effects coefficients:")
print(random_effects_results.params)
print("\nFirst-difference coefficients:")
print(first_difference_results.params)

Between OLS coefficients:
exper      0.454910
expersq   -0.029118
married    0.212258
union      0.195862
Name: parameter, dtype: float64

Random effects coefficients:
exper      0.334724
expersq   -0.018271
married    0.040066
union      0.183396
Name: parameter, dtype: float64

First-difference coefficients:
exper      0.115750
expersq   -0.003882
married    0.038138
union      0.042788
Name: parameter, dtype: float64


# <span style="color:darkblue"> VI. Practical Takeaways </span>

<font size = "5">

- Use `statsmodels` for a broad range of general-purpose statistical models and familiar formula-based OLS workflows.
- Consider `linearmodels` when you need panel estimators such as `PanelOLS` or other types of models such as instrumental variables.
- For panel models, set the entity-time MultiIndex correctly and choose effects and covariance estimators based on the sampling and research design.